# Scatter Plot Analysis

This notebook demonstrates `ScatterPlotterAnalysis` — a scan-level analyzer that
reads scalar columns from the s-file and produces a two-parameter scatter plot.

Key features covered here:
- Multiple y-series on twinned axes, each with its own color
- Optional `x_column`: any s-file column as the x-axis (falls back to the scan
  parameter when omitted)
- Optional `y_range` per series for fixed axis limits
- Optional `label` for readable axis / legend text
- Config-driven usage via `ScatterAnalyzerConfig` + `create_analyzer()`

Output lands at:
```
<date_dir>/analysis/Scan{NNN}/scatter_plots/<filename>.png
```

In [ ]:
from geecs_data_utils import ScanTag
from scan_analysis.analyzers.common.scatter_plotter_analysis import (
    PlotParameter,
    ScatterPlotterAnalysis,
)

# Identify the scan to analyze
scan_tag = ScanTag(year=2026, month=5, day=5, number=18, experiment="Undulator")

## Basic usage

Each `PlotParameter` describes one y-series: which s-file column to read, what
label to show on the axis and legend, and what color to use.

Here the scan parameter (`U_ModeImagerESP Position.Axis 1 Alias:ModeImager`) is
also an s-file column, so we pass it explicitly as `x_column` to get physical
units on the x-axis instead of a bin index.

In [ ]:
parameters = [
    PlotParameter(
        key_name="UC_ModeImager_x_rms",
        legend_label="x RMS",
        axis_label="x RMS (px)",
        color="royalblue",
    ),
    PlotParameter(
        key_name="UC_ModeImager_y_rms",
        legend_label="y RMS",
        axis_label="y RMS (px)",
        color="tomato",
    ),
    PlotParameter(
        key_name="UC_ModeImager_image_peak_value",
        legend_label="Peak",
        axis_label="Peak Value (counts)",
        color="seagreen",
    ),
]

analyzer = ScatterPlotterAnalysis(
    use_median=True,
    title="ModeImager Beam Properties",
    parameters=parameters,
    filename="mode_imager_rms",
    x_column="U_ModeImagerESP Position.Axis 1 Alias:ModeImager",
)

display_files = analyzer.run_analysis(scan_tag)
print(display_files)

## Fixing y-axis ranges

Pass `y_range=(min, max)` to a `PlotParameter` to pin the y-axis limits for that
series. Useful when comparing plots across scans or when one noisy shot would
otherwise dominate the autoscale.

In [ ]:
parameters_fixed = [
    PlotParameter(
        key_name="UC_ModeImager_x_rms",
        legend_label="x RMS",
        axis_label="x RMS (px)",
        color="royalblue",
        y_range=(0, 30),
    ),
    PlotParameter(
        key_name="UC_ModeImager_y_rms",
        legend_label="y RMS",
        axis_label="y RMS (px)",
        color="tomato",
        y_range=(0, 30),
    ),
]

analyzer_fixed = ScatterPlotterAnalysis(
    use_median=True,
    title="ModeImager RMS (fixed y-range)",
    parameters=parameters_fixed,
    filename="mode_imager_rms_fixed",
    x_column="U_ModeImagerESP Position.Axis 1 Alias:ModeImager",
)

analyzer_fixed.run_analysis(scan_tag)

## Default x-axis (scan parameter)

When `x_column` is omitted, the x-axis falls back to the scan parameter for 1D
scans, or shot number for noscans. This is the original behavior.

In [ ]:
analyzer_default_x = ScatterPlotterAnalysis(
    use_median=True,
    title="ModeImager RMS (default x-axis)",
    parameters=parameters[:2],
    filename="mode_imager_rms_default_x",
    # x_column omitted — uses scan parameter
)

analyzer_default_x.run_analysis(scan_tag)

## Config-driven usage

`ScatterAnalyzerConfig` is the Pydantic model that lets scatter plots be declared
in experiment YAML configs alongside `array2d` / `array1d` analyzers.
The `create_analyzer()` factory converts the config into a ready-to-use
`ScatterPlotterAnalysis` instance.

This is the preferred approach for production configs — the YAML equivalent is:

```yaml
- type: scatter
  title: "ModeImager Beam Properties"
  filename: "mode_imager_rms_config"
  x_column: "U_ModeImagerESP Position.Axis 1 Alias:ModeImager"
  use_median: true
  priority: 200
  parameters:
    - key_name: UC_ModeImager_x_rms
      label: "x RMS (px)"
      color: royalblue
      y_range: [0, 30]
    - key_name: UC_ModeImager_y_rms
      label: "y RMS (px)"
      color: tomato
      y_range: [0, 30]
    - key_name: UC_ModeImager_image_peak_value
      label: "Peak Value"
      color: seagreen
```

In [ ]:
from scan_analysis.config.analyzer_config_models import (
    PlotParameterConfig,
    ScatterAnalyzerConfig,
)
from scan_analysis.config.analyzer_factory import create_analyzer

cfg = ScatterAnalyzerConfig(
    title="ModeImager Beam Properties",
    filename="mode_imager_rms_config",
    x_column="U_ModeImagerESP Position.Axis 1 Alias:ModeImager",
    use_median=True,
    parameters=[
        PlotParameterConfig(
            key_name="UC_ModeImager_x_rms",
            label="x RMS (px)",
            color="royalblue",
            y_range=[0, 30],
        ),
        PlotParameterConfig(
            key_name="UC_ModeImager_y_rms",
            label="y RMS (px)",
            color="tomato",
            y_range=[0, 30],
        ),
        PlotParameterConfig(
            key_name="UC_ModeImager_image_peak_value",
            label="Peak Value",
            color="seagreen",
        ),
    ],
)

analyzer_from_config = create_analyzer(cfg)
display_files = analyzer_from_config.run_analysis(scan_tag)
print(display_files)